# Day 02b: Tokenization from Scratch

**Time:** ~20 minutes  
**Prerequisites:** Day 02a (What's Inside a Model)  
**Goal:** Build a BPE tokenizer by hand — understand how text becomes numbers.

---

In 02a we saw that `wte.weight` has shape `[50257, 768]` — that's 50,257 rows, one per token in the vocabulary. But what is a "token"? How does `"The cat sat"` become `[464, 9246, 3332]`?

Tokenization is the **front door** to the model. No neural network involved — it's just a lookup table and a set of merge rules.

## Why Not Just Use Characters?

There are three ways to split text into tokens. Each has tradeoffs — like choosing between microservices, monoliths, and serverless:

| Strategy | Tokens for "unhappiness" | Vocab Size | Problem |
|----------|--------------------------|------------|----------|
| **Characters** | `u, n, h, a, p, p, i, n, e, s, s` (11 tokens) | ~256 | Sequences are very long. Model wastes capacity learning spelling. |
| **Whole words** | `unhappiness` (1 token) | 500K+ | Vocab explodes. Can't handle new/rare words. OOV errors. |
| **Subwords (BPE)** | `un, happiness` (2 tokens) | 50K | Sweet spot. Common words = 1 token. Rare words = split into known pieces. |

```
Characters  = every request gets its own pod (lots of overhead)
Whole words = one giant monolith (can't handle edge cases)
BPE         = right-sized microservices (efficient + flexible)
```

GPT-2 uses **Byte Pair Encoding (BPE)** with a vocabulary of 50,257 tokens.

## Step 1: Load GPT-2's Vocabulary

GPT-2's tokenizer needs two files from the model directory:

- **`vocab.json`** — maps token strings to integer IDs (the lookup table)
- **`merges.txt`** — the BPE merge rules (how to combine bytes into tokens)

We downloaded both in notebook 01.

In [3]:
import json
import os

MODEL_DIR = "gpt2_weights"

with open(os.path.join(MODEL_DIR, "vocab.json")) as f:
    vocab = json.load(f)  # token string -> integer ID

with open(os.path.join(MODEL_DIR, "merges.txt")) as f:
    merges_raw = f.read().strip().split("\n")[1:]  # skip header line

# Reverse lookup: ID -> token string
id_to_token = {v: k for k, v in vocab.items()}

print(f"Vocabulary size: {len(vocab):,} tokens")
print(f"Merge rules:     {len(merges_raw):,}")
print()
print("Sample tokens (ID -> string):")
for token_id in [0, 1, 100, 464, 9246, 3332, 50256]:
    print(f"  {token_id:>6d}  ->  {repr(id_to_token[token_id])}")

Vocabulary size: 50,257 tokens
Merge rules:     50,000

Sample tokens (ID -> string):
       0  ->  '!'
       1  ->  '"'
     100  ->  '§'
     464  ->  'The'
    9246  ->  'cat'
    3332  ->  'Ġsat'
   50256  ->  '<|endoftext|>'


## What's With the Weird Characters?

You'll see tokens like `'Ġcat'` instead of `' cat'`. GPT-2 uses a byte-level encoding where:
- `Ġ` represents a space (byte 0x20)
- Other special chars map bytes that aren't printable ASCII

This is GPT-2's way of encoding **every possible byte** as a printable Unicode character. It means the tokenizer can handle any input — including binary data — without special "unknown token" handling.

Think of it as base64 encoding for the tokenizer's internal representation.

In [4]:
# GPT-2's byte-to-unicode mapping
def bytes_to_unicode():
    """Build GPT-2's byte <-> unicode character mapping.
    
    Maps each byte (0-255) to a printable Unicode character.
    Printable ASCII bytes map to themselves.
    Non-printable bytes get mapped to characters starting at U+0100.
    """
    # Bytes that are already printable ASCII
    bs = list(range(ord("!"), ord("~") + 1))  # 33-126
    bs += list(range(ord("\xa1"), ord("\xac") + 1))  # 161-172
    bs += list(range(ord("\xae"), ord("\xff") + 1))  # 174-255
    
    cs = bs[:]
    # Non-printable bytes get mapped to U+0100 onwards
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    
    return dict(zip(bs, [chr(c) for c in cs]))

byte_encoder = bytes_to_unicode()
byte_decoder = {v: k for k, v in byte_encoder.items()}

# Show the key mapping
print("Byte encoding examples:")
print(f"  Space (0x20) -> '{byte_encoder[32]}'  (this is the Ġ you see everywhere)")
print(f"  'a' (0x61)   -> '{byte_encoder[97]}'   (printable ASCII maps to itself)")
print(f"  'A' (0x41)   -> '{byte_encoder[65]}'   (same)")
print(f"  Tab (0x09)   -> '{byte_encoder[9]}'")
print(f"  Newline(0x0a)-> '{byte_encoder[10]}'")

Byte encoding examples:
  Space (0x20) -> 'Ġ'  (this is the Ġ you see everywhere)
  'a' (0x61)   -> 'a'   (printable ASCII maps to itself)
  'A' (0x41)   -> 'A'   (same)
  Tab (0x09)   -> 'ĉ'
  Newline(0x0a)-> 'Ċ'


## Step 2: How BPE Encoding Works

BPE encoding converts text to token IDs in three steps:

```
  "The cat"  
       │
       ▼  Step A: Convert each byte to GPT-2's unicode char
  ["T", "h", "e", "Ġ", "c", "a", "t"]
       │
       ▼  Step B: Repeatedly merge the highest-priority adjacent pair
  ["The", "Ġcat"]         (after many merges)
       │
       ▼  Step C: Look up each merged token in vocab.json
  [464, 9246]
```

The merge rules in `merges.txt` are ordered by priority (most common pairs first).  
This is like a compression algorithm — the most frequent byte pairs get merged first.

In [5]:
# Look at the first and last merge rules
print("First 10 merge rules (highest priority = most common pairs):")
for i, merge in enumerate(merges_raw[:10]):
    pair = tuple(merge.split())
    print(f"  {i:>5d}: '{pair[0]}' + '{pair[1]}' -> '{pair[0]}{pair[1]}'")

print(f"\nLast 5 merge rules (lowest priority):")
for i, merge in enumerate(merges_raw[-5:], len(merges_raw)-5):
    pair = tuple(merge.split())
    print(f"  {i:>5d}: '{pair[0]}' + '{pair[1]}' -> '{pair[0]}{pair[1]}'")

print(f"\nTotal merge rules: {len(merges_raw):,}")
print(f"\nThe first merge 'Ġ' + 't' means: the space-t combination")
print(f"is the most common adjacent pair in GPT-2's training data.")

First 10 merge rules (highest priority = most common pairs):
      0: 'Ġ' + 't' -> 'Ġt'
      1: 'Ġ' + 'a' -> 'Ġa'
      2: 'h' + 'e' -> 'he'
      3: 'i' + 'n' -> 'in'
      4: 'r' + 'e' -> 're'
      5: 'o' + 'n' -> 'on'
      6: 'Ġt' + 'he' -> 'Ġthe'
      7: 'e' + 'r' -> 'er'
      8: 'Ġ' + 's' -> 'Ġs'
      9: 'a' + 't' -> 'at'

Last 5 merge rules (lowest priority):
  49995: 'om' + 'inated' -> 'ominated'
  49996: 'Ġreg' + 'ress' -> 'Ġregress'
  49997: 'ĠColl' + 'ider' -> 'ĠCollider'
  49998: 'Ġinform' + 'ants' -> 'Ġinformants'
  49999: 'Ġg' + 'azed' -> 'Ġgazed'

Total merge rules: 50,000

The first merge 'Ġ' + 't' means: the space-t combination
is the most common adjacent pair in GPT-2's training data.


## Step 3: Build the Encoder

Now we implement BPE encoding. The algorithm:

1. Split the input text into individual unicode-encoded bytes
2. Find all adjacent pairs
3. Find the pair with the **lowest merge index** (highest priority)
4. Merge that pair everywhere it appears
5. Repeat from step 2 until no more merges apply
6. Look up each final token in the vocabulary

It's like a greedy compression pass — always merge the most common pattern first.

In [6]:
import re

# Build merge ranking: (pair) -> priority (lower = merge first)
bpe_ranks = {}
for i, merge in enumerate(merges_raw):
    pair = tuple(merge.split())
    bpe_ranks[pair] = i

# GPT-2's regex pattern for splitting text into preliminary chunks
# This splits on word boundaries, keeping spaces attached to the following word
GPT2_PATTERN = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?[^\s\w]+|\s+(?!\S)|\s+"""
)

def get_pairs(tokens):
    """Get all adjacent pairs from a list of tokens."""
    return set(zip(tokens[:-1], tokens[1:]))

def bpe_merge(tokens):
    """Repeatedly merge the highest-priority pair until no merges remain."""
    while True:
        pairs = get_pairs(tokens)
        if not pairs:
            break
        
        # Find the pair with the lowest rank (highest priority)
        best_pair = min(pairs, key=lambda p: bpe_ranks.get(p, float("inf")))
        
        if best_pair not in bpe_ranks:
            break  # No more known merges
        
        # Merge that pair everywhere in the token list
        first, second = best_pair
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == first and tokens[i+1] == second:
                new_tokens.append(first + second)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    
    return tokens

def encode(text):
    """Encode text to token IDs using GPT-2 BPE."""
    token_ids = []
    
    # Split text into preliminary chunks
    chunks = re.findall(GPT2_PATTERN, text)
    
    for chunk in chunks:
        # Convert each byte to GPT-2's unicode representation
        unicode_chars = [byte_encoder[b] for b in chunk.encode("utf-8")]
        
        # Apply BPE merges
        merged = bpe_merge(unicode_chars)
        
        # Look up each merged token in the vocabulary
        for token in merged:
            token_ids.append(vocab[token])
    
    return token_ids

print("Encoder built. Let's test it.")

Encoder built. Let's test it.


## Step 4: Test the Encoder

Let's encode some text and trace the process step by step.

In [7]:
# Encode and show results
test_cases = [
    "The cat sat",
    "Inference engineering makes AI apps fast.",
    "Kubernetes",
    "NVIDIA",
    "192.168.1.76",
    "unhappiness",
]

for text in test_cases:
    ids = encode(text)
    tokens = [id_to_token[i] for i in ids]
    print(f"Text:   {repr(text)}")
    print(f"IDs:    {ids}")
    print(f"Tokens: {tokens}")
    print(f"Count:  {len(ids)} tokens for {len(text)} chars")
    print()

Text:   'The cat sat'
IDs:    [464, 3797, 3332]
Tokens: ['The', 'Ġcat', 'Ġsat']
Count:  3 tokens for 11 chars

Text:   'Inference engineering makes AI apps fast.'
IDs:    [818, 4288, 8705, 1838, 9552, 6725, 3049, 13]
Tokens: ['In', 'ference', 'Ġengineering', 'Ġmakes', 'ĠAI', 'Ġapps', 'Ġfast', '.']
Count:  8 tokens for 41 chars

Text:   'Kubernetes'
IDs:    [42, 18478, 3262, 274]
Tokens: ['K', 'uber', 'net', 'es']
Count:  4 tokens for 10 chars

Text:   'NVIDIA'
IDs:    [38021]
Tokens: ['NVIDIA']
Count:  1 tokens for 6 chars

Text:   '192.168.1.76'
IDs:    [17477, 13, 14656, 13, 16, 13, 4304]
Tokens: ['192', '.', '168', '.', '1', '.', '76']
Count:  7 tokens for 12 chars

Text:   'unhappiness'
IDs:    [403, 71, 42661]
Tokens: ['un', 'h', 'appiness']
Count:  3 tokens for 11 chars



## Step 5: Watch BPE Merges Happen

Let's trace the merge process step by step for one word to see exactly how BPE works.

In [8]:
def bpe_merge_verbose(tokens):
    """Same as bpe_merge, but prints each step."""
    step = 0
    print(f"  Start: {tokens}")
    
    while True:
        pairs = get_pairs(tokens)
        if not pairs:
            break
        
        best_pair = min(pairs, key=lambda p: bpe_ranks.get(p, float("inf")))
        if best_pair not in bpe_ranks:
            break
        
        rank = bpe_ranks[best_pair]
        first, second = best_pair
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == first and tokens[i+1] == second:
                new_tokens.append(first + second)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
        step += 1
        print(f"  Merge {step}: '{first}' + '{second}' (rank {rank}) -> {tokens}")
    
    return tokens

# Trace "Kubernetes"
word = "Kubernetes"
print(f"Tracing BPE for: {repr(word)}")
print()
unicode_chars = [byte_encoder[b] for b in word.encode("utf-8")]
result = bpe_merge_verbose(unicode_chars)
print(f"\nFinal tokens: {result}")
print(f"Token IDs: {[vocab[t] for t in result]}")

Tracing BPE for: 'Kubernetes'

  Start: ['K', 'u', 'b', 'e', 'r', 'n', 'e', 't', 'e', 's']
  Merge 1: 'e' + 'r' (rank 7) -> ['K', 'u', 'b', 'er', 'n', 'e', 't', 'e', 's']
  Merge 2: 'e' + 's' (rank 18) -> ['K', 'u', 'b', 'er', 'n', 'e', 't', 'es']
  Merge 3: 'e' + 't' (rank 60) -> ['K', 'u', 'b', 'er', 'n', 'et', 'es']
  Merge 4: 'b' + 'er' (rank 271) -> ['K', 'u', 'ber', 'n', 'et', 'es']
  Merge 5: 'n' + 'et' (rank 3006) -> ['K', 'u', 'ber', 'net', 'es']
  Merge 6: 'u' + 'ber' (rank 18222) -> ['K', 'uber', 'net', 'es']

Final tokens: ['K', 'uber', 'net', 'es']
Token IDs: [42, 18478, 3262, 274]


## Step 6: Build the Decoder

Decoding is simpler — just reverse the lookup and convert bytes back to text.

In [9]:
def decode(token_ids):
    """Decode token IDs back to text."""
    # Look up each ID -> token string
    token_strings = [id_to_token[i] for i in token_ids]
    
    # Concatenate all token strings
    combined = "".join(token_strings)
    
    # Convert GPT-2 unicode chars back to bytes, then to text
    byte_values = [byte_decoder[c] for c in combined]
    return bytes(byte_values).decode("utf-8", errors="replace")

# Verify round-trip: encode then decode should give back the original
for text in test_cases:
    ids = encode(text)
    decoded = decode(ids)
    match = "OK" if decoded == text else "MISMATCH"
    print(f"  [{match}] {repr(text)} -> {ids} -> {repr(decoded)}")

  [OK] 'The cat sat' -> [464, 3797, 3332] -> 'The cat sat'
  [OK] 'Inference engineering makes AI apps fast.' -> [818, 4288, 8705, 1838, 9552, 6725, 3049, 13] -> 'Inference engineering makes AI apps fast.'
  [OK] 'Kubernetes' -> [42, 18478, 3262, 274] -> 'Kubernetes'
  [OK] 'NVIDIA' -> [38021] -> 'NVIDIA'
  [OK] '192.168.1.76' -> [17477, 13, 14656, 13, 16, 13, 4304] -> '192.168.1.76'
  [OK] 'unhappiness' -> [403, 71, 42661] -> 'unhappiness'


## Why Tokenization Matters for Inference

Tokenization affects inference performance directly:

| Factor | Impact |
|--------|--------|
| **Tokens per request** | More tokens = more forward passes = slower response |
| **Vocabulary size** | Larger vocab = larger embedding table = more GPU memory |
| **Token efficiency** | Better tokenizer = fewer tokens for same text = faster + cheaper |

This is why newer models (Llama 3, Qwen) use larger vocabularies (128K+) with more efficient tokenization. Fewer tokens per request = faster inference.

In [10]:
# Token efficiency: how many characters per token?
sample_text = (
    "The inference engine processes requests by loading model weights into GPU memory, "
    "tokenizing the input, running the forward pass through transformer blocks, and "
    "sampling output tokens autoregressively until the stop condition is met."
)

ids = encode(sample_text)
tokens = [id_to_token[i] for i in ids]

print(f"Text length:   {len(sample_text)} characters")
print(f"Token count:   {len(ids)} tokens")
print(f"Chars/token:   {len(sample_text) / len(ids):.1f}")
print(f"\nTokenized:")

# Show tokens with boundaries
line = ""
for t in tokens:
    decoded_t = bytes([byte_decoder[c] for c in t]).decode("utf-8", errors="replace")
    line += f"[{decoded_t}]"
print(f"  {line}")

Text length:   233 characters
Token count:   41 tokens
Chars/token:   5.7

Tokenized:
  [The][ inference][ engine][ processes][ requests][ by][ loading][ model][ weights][ into][ GPU][ memory][,][ token][izing][ the][ input][,][ running][ the][ forward][ pass][ through][ transformer][ blocks][,][ and][ sampling][ output][ tokens][ aut][ore][gress][ively][ until][ the][ stop][ condition][ is][ met][.]


## Key Takeaways

- Tokenization is **not** a neural network — it's a deterministic lookup + merge algorithm.
- GPT-2 uses **Byte Pair Encoding**: start with individual bytes, greedily merge the most common pairs.
- Common words like "the" become 1 token. Rare words like "Kubernetes" get split into subwords.
- The byte-level encoding (`Ġ` = space) means the tokenizer can handle **any input** without unknown tokens.
- Token count directly impacts inference cost — fewer tokens = fewer forward passes = faster.

## Next

**Notebook 03** — We use these token IDs to look up **embeddings** from `wte.weight` — converting integers into the 768-dimensional vectors that the model actually works with.

## References

- *Inference Engineering* Ch 2.2 (pp. 46–47) — Tokenization, Subword Tokenization
- Sennrich et al., "Neural Machine Translation of Rare Words with Subword Units" (2016) — BPE for NLP